In [ ]:
!pip install catboost

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)

In [ ]:
print(f"Shape: {df_delivery.shape}")
df_delivery.head()

In [ ]:
df_delivery.info()

In [ ]:
df_delivery.describe()

In [ ]:
print(f"delivery_time: {df_delivery['Delivery_Time'].sum()}")
print(f": {(df_delivery['Delivery_Time'] == 0).sum()}")

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_delivery)

In [ ]:
def check_duplicates(df):
  duplicates = df_delivery.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery)

In [ ]:
features = df_delivery.columns.drop("Order_ID")

In [ ]:
categorical_cols = df_delivery.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))


In [ ]:
def check_target_distribution(df_delivery, Delivery_Time):
  df_delivery[Delivery_Time].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({Delivery_Time})")
  plt.xlabel(Delivery_Time)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_delivery, "Delivery_Time")

In [ ]:
df_delivery.describe()

In [ ]:
feature_cols = ['Order_ID', 'Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs',
                'Delivery_Time']
X = df_delivery[feature_cols]
y = df_delivery['Delivery_Time']


In [ ]:
models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
  "Support Vector Machine": SVR(kernel='rbf'),
  "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10),
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "LightGBM": LGBMRegressor(verbose=-1),

}

In [ ]:
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

In [ ]:

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols ,
    'importance': model.feature_clos_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['speed'].dropna(), bins=30, edgecolor='black', color='orange')
plt.title('delivery time')
plt.xlabel('delivery')
plt.ylabel('time')
plt.show()

In [ ]:
# Task Bonus: Write your code here: